In [1]:
import pandas as pd

df = pd.read_csv("prelabelled_dataset.csv")

llm_df = df[df["needs_llm"] == 1][["article_id", "title", "text"]].copy()

print("Rows to send to Mistral:", len(llm_df))
llm_df.head(3)

Rows to send to Mistral: 10813


,article_id,title,text
7,ee5c979737eabb9aff1b3b9a344ee1a500e3bad0,"HDFC Bank governance strong, depositors have n...",Depositors have no reason to worry as HDFC Ban...
11,7602312d25a54872f135c69c77bf1acadc4e697a,Japan Industrial Output Grows More Than Estima...,-\nJapan Industrial Output Grows More Than Est...
20,5270544ba069185841ca8279d8191bcbee937885,Allworth Financial LP Raises Position in iShar...,Allworth Financial LP increased its stake in i...


In [2]:
test_df = llm_df.head(5).copy()
test_df

,article_id,title,text
7,ee5c979737eabb9aff1b3b9a344ee1a500e3bad0,"HDFC Bank governance strong, depositors have n...",Depositors have no reason to worry as HDFC Ban...
11,7602312d25a54872f135c69c77bf1acadc4e697a,Japan Industrial Output Grows More Than Estima...,-\nJapan Industrial Output Grows More Than Est...
20,5270544ba069185841ca8279d8191bcbee937885,Allworth Financial LP Raises Position in iShar...,Allworth Financial LP increased its stake in i...
23,f0cde1ef2e854e27c7cc37a1e79c6bf0ee145d7b,"VK increases revenue 8% to 160 bln rubles, adj...",19 Mar 2026 10:19 VK increases revenue 8% to 1...
27,994a7da3c3b3bbb8544786982b20bed508ead3a1,India Gold Price Today: Gold Rises Significant...,In a notable market movement reported on April...


In [9]:
# build the prompt

In [22]:
def build_prompt(title, text):
    return f"""
Classify this financial news article into 3 tag lists.

Return ONLY valid JSON.
Do not use markdown.
Do not explain.
Do not include generic labels.
Do not include category names or category descriptions as tags.

Important:
- Tags must be specific topic or entity names only
- Do NOT output tags like "economy-wide themes", "sector themes", "industry themes", "specific companies", "macro", "industry", or "entity"
- Each field must contain only a list of short strings
- If there is no valid tag for a field, return []

Use exactly this JSON schema:
{{
  "macro_tags": [],
  "industry_tags": [],
  "entity_tags": []
}}

Examples of good macro tags:
["inflation", "interest rates", "GDP", "Federal Reserve", "recession"]

Examples of good industry tags:
["banking", "semiconductors", "airlines", "real estate", "insurance"]

Examples of good entity tags:
["Goldman Sachs", "JPMorgan", "European Central Bank", "Tesla"]

Article title:
{title}

Article text:
{text[:2000]}
"""

In [8]:
# make Sample prompt

In [4]:
sample_prompt = build_prompt(test_df.iloc[0]["title"], test_df.iloc[0]["text"])
print(sample_prompt[:3000])


You are labeling a financial news article for a financial intelligence dataset.

Task:
Read the article and extract tags for these 3 levels:
1. Macro = economy-wide factors such as inflation, interest rates, GDP, central banks, recession, fiscal policy, monetary policy, trade, regulation
2. Industry = sector or industry-level themes such as banking, semiconductors, airlines, real estate, retail, insurance, energy, telecommunications
3. Entity = specific companies, institutions, funds, regulators, or named organizations

Rules:
- An article may have tags in more than one level
- Return JSON only
- Do not explain your reasoning
- If no tags exist for one level, return an empty list

Output format:
{
  "macro_tags": [],
  "industry_tags": [],
  "entity_tags": []
}

Article title:
HDFC Bank governance strong, depositors have no reason to worry, say experts

Article text:
Depositors have no reason to worry as HDFC Bank is well capitalised with no indications of financial stress or liquidit

In [7]:
# Call mistral thru Ollama

In [5]:
import subprocess

def call_mistral(prompt, model="mistral"):
    result = subprocess.run(
        ["ollama", "run", model, prompt],
        capture_output=True,
        text=True,
        encoding="utf-8"
    )
    return result.stdout.strip()

In [6]:
# Test mistral on one row

In [10]:
response = call_mistral(
    build_prompt(test_df.iloc[0]["title"], test_df.iloc[0]["text"])
)

print(response)

{
  "macro_tags": ["central banks", "regulation"],
  "industry_tags": ["banking"],
  "entity_tags": ["HDFC Bank", "Atanu Chakraborty", "Keki Mistry", "Reserve Bank of India", "Aditya Bhattacharya", "King Stubb and Kasiva"]
}


In [11]:
# parse json output

In [18]:
import json
import re

def parse_mistral_output(output):
    try:
        return json.loads(output)
    except Exception:
        pass

    try:
        match = re.search(r"\{.*\}", output, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception:
        pass

    return {
        "macro_tags": [],
        "industry_tags": [],
        "entity_tags": [],
        "raw_output": output
    }

In [14]:
# test
parsed = parse_mistral_output(response)
parsed

{'macro_tags': ['central banks', 'regulation'],
 'industry_tags': ['banking'],
 'entity_tags': ['HDFC Bank',
  'Atanu Chakraborty',
  'Keki Mistry',
  'Reserve Bank of India',
  'Aditya Bhattacharya',
  'King Stubb and Kasiva']}

In [21]:
#try for 5 rows
results = []

for _, row in test_df.iterrows():
    prompt = build_prompt(row["title"], row["text"])
    output = call_mistral(prompt)
    parsed = parse_mistral_output(output)

    results.append({
        "article_id": row["article_id"],
        "macro_tags": parsed.get("macro_tags", []),
        "industry_tags": parsed.get("industry_tags", []),
        "entity_tags": parsed.get("entity_tags", []),
        "raw_output": output
    })

results_df = pd.DataFrame(results)
results_df

,article_id,macro_tags,industry_tags,entity_tags,raw_output
0,ee5c979737eabb9aff1b3b9a344ee1a500e3bad0,[],[banking],"[HDFC Bank, Atanu Chakraborty, Keki Mistry]","{\n ""macro_tags"": [],\n ""industry_tags"": [""b..."
1,7602312d25a54872f135c69c77bf1acadc4e697a,"[Japan's industrial production, industrial out...",[manufacturing],"[Ministry of Economy, Trade and Industry (METI)]","{\n ""macro_tags"": [""Japan's industrial produc..."
2,5270544ba069185841ca8279d8191bcbee937885,[],[bond_etfs],"[iShares Core U.S. Aggregate Bond ETF, Allwort...","{\n ""macro_tags"": [],\n ""industry_tags"": [""b..."
3,f0cde1ef2e854e27c7cc37a1e79c6bf0ee145d7b,"[revenue growth, profit margins]","[technology, holding companies]",[VK],"{\n ""macro_tags"": [""revenue growth"", ""profit ..."
4,994a7da3c3b3bbb8544786982b20bed508ead3a1,"[inflation, commodities]",[gold],[India],"{\n ""macro_tags"": [""inflation"", ""commodities""..."


In [16]:
results_df.to_csv("mistral_labelled_test_batch.csv", index=False)
print("Saved mistral_labelled_test_batch.csv")

Saved mistral_labelled_test_batch.csv


In [24]:
# 10 row batch
test_10_df = llm_df.head(10).copy()

results = []

for _, row in test_10_df.iterrows():
    prompt = build_prompt(row["title"], row["text"])
    output = call_mistral(prompt)
    parsed = parse_mistral_output(output)

    results.append({
        "article_id": row["article_id"],
        "macro_tags": parsed.get("macro_tags", []),
        "industry_tags": parsed.get("industry_tags", []),
        "entity_tags": parsed.get("entity_tags", []),
        "raw_output": output
    })

results_10_df = pd.DataFrame(results)
results_10_df

,article_id,macro_tags,industry_tags,entity_tags,raw_output
0,ee5c979737eabb9aff1b3b9a344ee1a500e3bad0,[],[banking],"[HDFC Bank, Atanu Chakraborty, Keki Mistry, Re...","{\n ""macro_tags"": [],\n ""industry_tags"": [""b..."
1,7602312d25a54872f135c69c77bf1acadc4e697a,"[industrial production, growth rate, economic ...","[manufacturing, Japanese industries]","[Ministry of Economy, Trade and Industry (METI)]","{\n ""macro_tags"": [""industrial production"", ""..."
2,5270544ba069185841ca8279d8191bcbee937885,[],"[bond_etfs, securities_exchange]","[Allworth Financial LP, iShares Core U.S. Aggr...","{\n ""macro_tags"": [],\n ""industry_tags"": [""b..."
3,f0cde1ef2e854e27c7cc37a1e79c6bf0ee145d7b,[],"[technology, holding]",[VK],"{\n ""macro_tags"": [],\n ""industry_tags"": [""t..."
4,994a7da3c3b3bbb8544786982b20bed508ead3a1,"[inflation, commodities, market movements]","[gold mining, precious metals]",[Bitcoin World],"{\n ""macro_tags"": [""inflation"", ""commodities""..."
5,9a846c2dd53d7f0cc960277d837177eb45e1574d,[],[airlines],"[Achmea Investment Management B.V., United Air...","{\n ""macro_tags"": [],\n ""industry_tags"": [""a..."
6,08314ebf6f46647b6ad97bc78556f12b7b77bc7b,[],[cryptocurrency],"[Binance, Tether, Bitcoin]","{\n ""macro_tags"": [],\n ""industry_tags"": [""c..."
7,fe5a7f12ada55908b4075ea4026688b828b27992,"[direct tax collections, corporate tax, non-co...",[],[],"{\n ""macro_tags"": [""direct tax collections"", ..."
8,fab7cbb4d8c06a27a2019cc5bc8c943d12eb8f48,[],[agriculture],"[Pradhan Mantri Kisan Samman Nidhi, Narendra M...","{\n ""macro_tags"": [],\n ""industry_tags"": [""a..."
9,9b6a14279003e2b1a648e094fab2c3772277fa9c,"[Middle East conflict, energy prices, global i...","[renewable energy, oil and gas, infrastructure...","[World Bank, International Finance Corporation...","{\n ""macro_tags"": [""Middle East conflict"", ""e..."


In [25]:
test_50_df = llm_df.head(50).copy()

results = []

for _, row in test_50_df.iterrows():
    prompt = build_prompt(row["title"], row["text"])
    output = call_mistral(prompt)
    parsed = parse_mistral_output(output)

    results.append({
        "article_id": row["article_id"],
        "macro_tags": parsed.get("macro_tags", []),
        "industry_tags": parsed.get("industry_tags", []),
        "entity_tags": parsed.get("entity_tags", []),
        "raw_output": output
    })

results_50_df = pd.DataFrame(results)
results_50_df

,article_id,macro_tags,industry_tags,entity_tags,raw_output
0,ee5c979737eabb9aff1b3b9a344ee1a500e3bad0,[],[banking],"[HDFC Bank, Atanu Chakraborty, Keki Mistry, Re...","{\n ""macro_tags"": [],\n ""industry_tags"": [""b..."
1,7602312d25a54872f135c69c77bf1acadc4e697a,"[industrial production, GDP]",[manufacturing],"[Ministry of Economy, Trade and Industry]","{\n ""macro_tags"": [""industrial production"", ""..."
2,5270544ba069185841ca8279d8191bcbee937885,[],"[bond_etf, index_funds]","[Allworth Financial LP, iShares Core U.S. Aggr...","{\n ""macro_tags"": [],\n ""industry_tags"": [""b..."
3,f0cde1ef2e854e27c7cc37a1e79c6bf0ee145d7b,[],"[technology, holding]",[VK],"{\n ""macro_tags"": [],\n ""industry_tags"": [""t..."
4,994a7da3c3b3bbb8544786982b20bed508ead3a1,"[inflation, commodities]",[gold],[India],"{\n ""macro_tags"": [""inflation"", ""commodities""..."
5,9a846c2dd53d7f0cc960277d837177eb45e1574d,[],[airlines],"[Achmea Investment Management B.V., United Air...","{\n ""macro_tags"": [],\n ""industry_tags"": [""a..."
6,08314ebf6f46647b6ad97bc78556f12b7b77bc7b,[],[cryptocurrency],"[Binance, Tether]","{\n ""macro_tags"": [],\n ""industry_tags"": [""c..."
7,fe5a7f12ada55908b4075ea4026688b828b27992,"[direct tax collections, fiscal year 2025-26, ...",[],[],"{\n ""macro_tags"": [""direct tax collections"", ..."
8,fab7cbb4d8c06a27a2019cc5bc8c943d12eb8f48,[],[agriculture],"[Pradhan Mantri Kisan Samman Nidhi, Narendra M...","{\n ""macro_tags"": [],\n ""industry_tags"": [""a..."
9,9b6a14279003e2b1a648e094fab2c3772277fa9c,"[Middle East conflict, global instability, oil...","[renewable energy, solar power]","[World Bank, International Finance Corporation...","{\n ""macro_tags"": [""Middle East conflict"", ""g..."


In [26]:
import pandas as pd
import time

start_idx = 0
end_idx = 2000

batch_df = llm_df.iloc[start_idx:end_idx].copy()
results = []

start_time = time.time()

for i, (_, row) in enumerate(batch_df.iterrows(), start=1):
    try:
        prompt = build_prompt(row["title"], row["text"])
        output = call_mistral(prompt)
        parsed = parse_mistral_output(output)

        results.append({
            "article_id": row["article_id"],
            "macro_tags": parsed.get("macro_tags", []),
            "industry_tags": parsed.get("industry_tags", []),
            "entity_tags": parsed.get("entity_tags", []),
            "raw_output": output
        })

        global_row_num = start_idx + i
        elapsed = time.time() - start_time
        avg_per_row = elapsed / i
        remaining_rows = len(batch_df) - i
        est_remaining_sec = avg_per_row * remaining_rows

        print(f"Finished global row {global_row_num}/{len(llm_df)}")
        print(f"Elapsed: {elapsed/60:.2f} min | Avg/row: {avg_per_row:.2f} sec | Est remaining: {est_remaining_sec/60:.2f} min")

        if i % 10 == 0:
            pd.DataFrame(results).to_csv(
                f"mistral_progress_{start_idx}_{end_idx}.csv",
                index=False
            )
            print(f"Checkpoint saved at batch row {i}")

    except Exception as e:
        results.append({
            "article_id": row["article_id"],
            "macro_tags": [],
            "industry_tags": [],
            "entity_tags": [],
            "raw_output": f"ERROR: {str(e)}"
        })

        global_row_num = start_idx + i
        elapsed = time.time() - start_time
        avg_per_row = elapsed / i
        remaining_rows = len(batch_df) - i
        est_remaining_sec = avg_per_row * remaining_rows

        print(f"Error at global row {global_row_num}: {e}")
        print(f"Elapsed: {elapsed/60:.2f} min | Avg/row: {avg_per_row:.2f} sec | Est remaining: {est_remaining_sec/60:.2f} min")

        if i % 10 == 0:
            pd.DataFrame(results).to_csv(
                f"mistral_progress_{start_idx}_{end_idx}.csv",
                index=False
            )
            print(f"Checkpoint saved at batch row {i}")

results_df = pd.DataFrame(results)
results_df.to_csv(f"mistral_labelled_{start_idx}_{end_idx}.csv", index=False)

total_time = time.time() - start_time
print(f"Saved mistral_labelled_{start_idx}_{end_idx}.csv")
print(f"Total runtime: {total_time/60:.2f} minutes")
print(f"Average time per row: {total_time/len(batch_df):.2f} seconds")

Finished global row 1/10813
Elapsed: 0.20 min | Avg/row: 12.28 sec | Est remaining: 409.18 min
Finished global row 2/10813
Elapsed: 0.32 min | Avg/row: 9.56 sec | Est remaining: 318.24 min
Finished global row 3/10813
Elapsed: 0.48 min | Avg/row: 9.66 sec | Est remaining: 321.62 min
Finished global row 4/10813
Elapsed: 0.56 min | Avg/row: 8.43 sec | Est remaining: 280.58 min
Finished global row 5/10813
Elapsed: 0.64 min | Avg/row: 7.72 sec | Est remaining: 256.58 min
Finished global row 6/10813
Elapsed: 0.85 min | Avg/row: 8.52 sec | Est remaining: 283.00 min
Finished global row 7/10813
Elapsed: 0.98 min | Avg/row: 8.36 sec | Est remaining: 277.81 min
Finished global row 8/10813
Elapsed: 1.13 min | Avg/row: 8.48 sec | Est remaining: 281.55 min
Finished global row 9/10813
Elapsed: 1.28 min | Avg/row: 8.54 sec | Est remaining: 283.25 min
Finished global row 10/10813
Elapsed: 1.44 min | Avg/row: 8.64 sec | Est remaining: 286.56 min
Checkpoint saved at batch row 10
Finished global row 11/10

In [27]:
import pandas as pd
import time

start_idx = 2000
end_idx = 5000

batch_df = llm_df.iloc[start_idx:end_idx].copy()
results = []

start_time = time.time()

for i, (_, row) in enumerate(batch_df.iterrows(), start=1):
    try:
        prompt = build_prompt(row["title"], row["text"])
        output = call_mistral(prompt)
        parsed = parse_mistral_output(output)

        results.append({
            "article_id": row["article_id"],
            "macro_tags": parsed.get("macro_tags", []),
            "industry_tags": parsed.get("industry_tags", []),
            "entity_tags": parsed.get("entity_tags", []),
            "raw_output": output
        })

        # save checkpoint every 10 rows
        if i % 10 == 0:
            pd.DataFrame(results).to_csv(
                f"mistral_progress_{start_idx}_{end_idx}.csv",
                index=False
            )

        # print progress every 100 rows
        if i % 100 == 0:
            global_row_num = start_idx + i
            elapsed = time.time() - start_time
            avg_per_row = elapsed / i
            remaining_rows = len(batch_df) - i
            est_remaining_sec = avg_per_row * remaining_rows

            print(f"Finished global row {global_row_num}/{len(llm_df)}")
            print(f"Elapsed: {elapsed/60:.2f} min | Avg/row: {avg_per_row:.2f} sec | Est remaining: {est_remaining_sec/60:.2f} min")

    except Exception as e:
        results.append({
            "article_id": row["article_id"],
            "macro_tags": [],
            "industry_tags": [],
            "entity_tags": [],
            "raw_output": f"ERROR: {str(e)}"
        })

        print(f"Error at batch row {i}: {e}")

        if i % 10 == 0:
            pd.DataFrame(results).to_csv(
                f"mistral_progress_{start_idx}_{end_idx}.csv",
                index=False
            )

results_df = pd.DataFrame(results)
results_df.to_csv(f"mistral_labelled_{start_idx}_{end_idx}.csv", index=False)

total_time = time.time() - start_time
print(f"Saved mistral_labelled_{start_idx}_{end_idx}.csv")
print(f"Total runtime: {total_time/60:.2f} minutes")
print(f"Average time per row: {total_time/len(batch_df):.2f} seconds")

Finished global row 2100/10813
Elapsed: 9.77 min | Avg/row: 5.86 sec | Est remaining: 283.36 min
Finished global row 2200/10813
Elapsed: 19.20 min | Avg/row: 5.76 sec | Est remaining: 268.75 min
Finished global row 2300/10813
Elapsed: 28.42 min | Avg/row: 5.68 sec | Est remaining: 255.82 min
Finished global row 2400/10813
Elapsed: 37.43 min | Avg/row: 5.61 sec | Est remaining: 243.27 min
Finished global row 2500/10813
Elapsed: 46.75 min | Avg/row: 5.61 sec | Est remaining: 233.76 min
Finished global row 2600/10813
Elapsed: 56.19 min | Avg/row: 5.62 sec | Est remaining: 224.77 min
Finished global row 2700/10813
Elapsed: 65.91 min | Avg/row: 5.65 sec | Est remaining: 216.56 min
Finished global row 2800/10813
Elapsed: 76.38 min | Avg/row: 5.73 sec | Est remaining: 210.05 min
Finished global row 2900/10813
Elapsed: 87.85 min | Avg/row: 5.86 sec | Est remaining: 204.98 min
Finished global row 3000/10813
Elapsed: 98.07 min | Avg/row: 5.88 sec | Est remaining: 196.15 min
Finished global row 3

In [28]:
import pandas as pd
import time

start_idx = 5000
end_idx = 7000

batch_df = llm_df.iloc[start_idx:end_idx].copy()
results = []

start_time = time.time()

for i, (_, row) in enumerate(batch_df.iterrows(), start=1):
    try:
        prompt = build_prompt(row["title"], row["text"])
        output = call_mistral(prompt)
        parsed = parse_mistral_output(output)

        results.append({
            "article_id": row["article_id"],
            "macro_tags": parsed.get("macro_tags", []),
            "industry_tags": parsed.get("industry_tags", []),
            "entity_tags": parsed.get("entity_tags", []),
            "raw_output": output
        })

        # save checkpoint every 10 rows
        if i % 10 == 0:
            pd.DataFrame(results).to_csv(
                f"mistral_progress_{start_idx}_{end_idx}.csv",
                index=False
            )

        # print progress every 100 rows
        if i % 100 == 0:
            global_row_num = start_idx + i
            elapsed = time.time() - start_time
            avg_per_row = elapsed / i
            remaining_rows = len(batch_df) - i
            est_remaining_sec = avg_per_row * remaining_rows

            print(f"Finished global row {global_row_num}/{len(llm_df)}")
            print(f"Elapsed: {elapsed/60:.2f} min | Avg/row: {avg_per_row:.2f} sec | Est remaining: {est_remaining_sec/60:.2f} min")

    except Exception as e:
        results.append({
            "article_id": row["article_id"],
            "macro_tags": [],
            "industry_tags": [],
            "entity_tags": [],
            "raw_output": f"ERROR: {str(e)}"
        })

        print(f"Error at batch row {i}: {e}")

        if i % 10 == 0:
            pd.DataFrame(results).to_csv(
                f"mistral_progress_{start_idx}_{end_idx}.csv",
                index=False
            )

results_df = pd.DataFrame(results)
results_df.to_csv(f"mistral_labelled_{start_idx}_{end_idx}.csv", index=False)

total_time = time.time() - start_time
print(f"Saved mistral_labelled_{start_idx}_{end_idx}.csv")
print(f"Total runtime: {total_time/60:.2f} minutes")
print(f"Average time per row: {total_time/len(batch_df):.2f} seconds")

Finished global row 5100/10813
Elapsed: 10.04 min | Avg/row: 6.02 sec | Est remaining: 190.76 min
Finished global row 5200/10813
Elapsed: 20.52 min | Avg/row: 6.16 sec | Est remaining: 184.66 min
Finished global row 5300/10813
Elapsed: 32.42 min | Avg/row: 6.48 sec | Est remaining: 183.71 min
Finished global row 5400/10813
Elapsed: 43.02 min | Avg/row: 6.45 sec | Est remaining: 172.07 min
Finished global row 5500/10813
Elapsed: 53.76 min | Avg/row: 6.45 sec | Est remaining: 161.29 min
Finished global row 5600/10813
Elapsed: 63.91 min | Avg/row: 6.39 sec | Est remaining: 149.13 min
Finished global row 5700/10813
Elapsed: 73.93 min | Avg/row: 6.34 sec | Est remaining: 137.30 min
Finished global row 5800/10813
Elapsed: 84.08 min | Avg/row: 6.31 sec | Est remaining: 126.12 min
Finished global row 5900/10813
Elapsed: 94.74 min | Avg/row: 6.32 sec | Est remaining: 115.79 min
Finished global row 6000/10813
Elapsed: 104.29 min | Avg/row: 6.26 sec | Est remaining: 104.29 min
Finished global row

In [29]:
import pandas as pd
import time

start_idx = 7000
end_idx = 10813

batch_df = llm_df.iloc[start_idx:end_idx].copy()
results = []

start_time = time.time()

for i, (_, row) in enumerate(batch_df.iterrows(), start=1):
    try:
        prompt = build_prompt(row["title"], row["text"])
        output = call_mistral(prompt)
        parsed = parse_mistral_output(output)

        results.append({
            "article_id": row["article_id"],
            "macro_tags": parsed.get("macro_tags", []),
            "industry_tags": parsed.get("industry_tags", []),
            "entity_tags": parsed.get("entity_tags", []),
            "raw_output": output
        })

        # save checkpoint every 10 rows
        if i % 10 == 0:
            pd.DataFrame(results).to_csv(
                f"mistral_progress_{start_idx}_{end_idx}.csv",
                index=False
            )

        # print progress every 100 rows
        if i % 100 == 0:
            global_row_num = start_idx + i
            elapsed = time.time() - start_time
            avg_per_row = elapsed / i
            remaining_rows = len(batch_df) - i
            est_remaining_sec = avg_per_row * remaining_rows

            print(f"Finished global row {global_row_num}/{len(llm_df)}")
            print(f"Elapsed: {elapsed/60:.2f} min | Avg/row: {avg_per_row:.2f} sec | Est remaining: {est_remaining_sec/60:.2f} min")

    except Exception as e:
        results.append({
            "article_id": row["article_id"],
            "macro_tags": [],
            "industry_tags": [],
            "entity_tags": [],
            "raw_output": f"ERROR: {str(e)}"
        })

        print(f"Error at batch row {i}: {e}")

        if i % 10 == 0:
            pd.DataFrame(results).to_csv(
                f"mistral_progress_{start_idx}_{end_idx}.csv",
                index=False
            )

results_df = pd.DataFrame(results)
results_df.to_csv(f"mistral_labelled_{start_idx}_{end_idx}.csv", index=False)

total_time = time.time() - start_time
print(f"Saved mistral_labelled_{start_idx}_{end_idx}.csv")
print(f"Total runtime: {total_time/60:.2f} minutes")
print(f"Average time per row: {total_time/len(batch_df):.2f} seconds")

Finished global row 7100/10813
Elapsed: 11.71 min | Avg/row: 7.03 sec | Est remaining: 434.75 min
Finished global row 7200/10813
Elapsed: 22.96 min | Avg/row: 6.89 sec | Est remaining: 414.70 min
Finished global row 7300/10813
Elapsed: 34.54 min | Avg/row: 6.91 sec | Est remaining: 404.52 min
Finished global row 7400/10813
Elapsed: 46.50 min | Avg/row: 6.98 sec | Est remaining: 396.77 min
Finished global row 7500/10813
Elapsed: 57.42 min | Avg/row: 6.89 sec | Est remaining: 380.47 min
Finished global row 7600/10813
Elapsed: 66.78 min | Avg/row: 6.68 sec | Est remaining: 357.61 min
Finished global row 7700/10813
Elapsed: 76.36 min | Avg/row: 6.54 sec | Est remaining: 339.57 min
Finished global row 7800/10813
Elapsed: 87.39 min | Avg/row: 6.55 sec | Est remaining: 329.13 min
Finished global row 7900/10813
Elapsed: 97.03 min | Avg/row: 6.47 sec | Est remaining: 314.06 min
Finished global row 8000/10813
Elapsed: 106.55 min | Avg/row: 6.39 sec | Est remaining: 299.73 min
Finished global row

In [30]:
# Merge all labelled data

In [31]:
df1 = pd.read_csv("mistral_labelled_0_2000.csv")
df2 = pd.read_csv("mistral_labelled_2000_5000.csv")
df3 = pd.read_csv("mistral_labelled_5000_7000.csv")
df4 = pd.read_csv("mistral_labelled_7000_10813.csv")

final_labels_df = pd.concat([df1, df2, df3, df4], ignore_index=True)

print("Merged labelled rows:", len(final_labels_df))
print(final_labels_df.head())

final_labels_df.to_csv("mistral_all_labels.csv", index=False)
print("Saved mistral_all_labels.csv")

Merged labelled rows: 10813
                                 article_id  \
0  ee5c979737eabb9aff1b3b9a344ee1a500e3bad0   
1  7602312d25a54872f135c69c77bf1acadc4e697a   
2  5270544ba069185841ca8279d8191bcbee937885   
3  f0cde1ef2e854e27c7cc37a1e79c6bf0ee145d7b   
4  994a7da3c3b3bbb8544786982b20bed508ead3a1   

                                          macro_tags  \
0                                                 []   
1           ['industrial production', 'growth rate']   
2  ['securities_exchange', 'institutional_investm...   
3                                                 []   
4    ['inflation', 'commodities', 'market movement']   

               industry_tags  \
0                ['banking']   
1          ['manufacturing']   
2            ['index_funds']   
3  ['technology', 'holding']   
4                   ['gold']   

                                         entity_tags  \
0  ['HDFC Bank', 'Atanu Chakraborty', 'Keki Mistry']   
1  ['Ministry of Economy, Trade and Industry (M

In [32]:
print("Duplicate article_id count:", final_labels_df["article_id"].duplicated().sum())

Duplicate article_id count: 0


In [33]:
# Merge back to cleaned dataset

In [34]:
cleaned_df = pd.read_csv("raw_ingested_articles_cleaned.csv")
labels_df = pd.read_csv("mistral_all_labels.csv")

silver_df = cleaned_df.merge(
    labels_df[["article_id", "macro_tags", "industry_tags", "entity_tags"]],
    on="article_id",
    how="left"
)

print("Silver rows:", len(silver_df))
print(silver_df.head())

silver_df.to_csv("silver_dataset_final.csv", index=False)
print("Saved silver_dataset_final.csv")

Silver rows: 62203
                                 article_id  \
0  71e3502a665cb219fbf1180ffa8e3d83e48d0a5f   
1  7e1f5b18780831791cf76ed3ed83620a5dd299dd   
2  3ba4c8cebb884f3c37afb91d82b3c220b6a16c51   
3  8bf04d9e8e0e8ff7ed5ccd40da0f269bdae3af7f   
4  55b3f91372c0eed8acd2759be6724bd4c273f84a   

                                               title  \
0  Philippine Peso Hits Record Lows: Is Now the B...   
1  Power Assets Earnings: Eyes on Use of UKPN Sal...   
2  Oil rises after Iran strikes Middle East energ...   
3  The United States and Japan have agreed to inv...   
4  AIA Group Ltd Has $9.68 Million Position in Th...   

                                                text  \
0  Dubai: Filipino expats in the UAE are seeing s...   
1  Power Assets Holdings Ltd\n00006: XHKG (HKG)\n...   
2  BEIJING: Oil prices rose on Thursday, with ben...   
3  The United States and Japan have agreed to inv...   
4  AIA Group Ltd Has $9.68 Million Position in Th...   

                    publ

In [35]:
print("Rows with macro tags:", silver_df["macro_tags"].notna().sum())
print("Rows with industry tags:", silver_df["industry_tags"].notna().sum())
print("Rows with entity tags:", silver_df["entity_tags"].notna().sum())

Rows with macro tags: 10813
Rows with industry tags: 10813
Rows with entity tags: 10813
